<a href="https://colab.research.google.com/github/abdelrahman-rajab-hassan/machine-learning-lab/blob/main/supervised-machine-learning/project-1-part-6-(core)/Machine_Learning_Project_For_Sales_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Building a machine learning model to predict the Item_outlet_sales

### Prjoect 1 - Part 5 (Core)

# Sales prediction data dictionary

| Variable Name               | Description                                                                 |
|----------------------------|-----------------------------------------------------------------------------|
| Item_Identifier            | Unique product ID                                                          |
| Item_Weight                | Weight of product                                                          |
| Item_Fat_Content           | Whether the product is low fat or regular                                  |
| Item_Visibility            | The percentage of total display area of all products in a store allocated to the particular product |
| Item_Type                  | The category to which the product belongs                                  |
| Item_MRP                   | Maximum Retail Price (list price) of the product                           |
| Outlet_Identifier          | Unique store ID                                                            |
| Outlet_Establishment_Year  | The year in which store was established                                    |
| Outlet_Size                | The size of the store in terms of ground area covered                      |
| Outlet_Location_Type       | The type of area in which the store is located                             |
| Outlet_Type                | Whether the outlet is a grocery store or some sort of supermarket          |
| Item_Outlet_Sales          | Sales of the product in the particular store. This is the target variable to be predicted |

In [ ]:
# import necessary libraries
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, root_mean_squared_error, r2_score



from sklearn import set_config

set_config(transform_output = 'pandas')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

path = '/content/drive/MyDrive/data_science_projects/sales_predictions_2023 (1).csv'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Check for dataframe
df = pd.read_csv(path)
df.head()

,Item_Identifier,Item_Weight,Item_Fat_Content,Item_Visibility,Item_Type,Item_MRP,Outlet_Identifier,Outlet_Establishment_Year,Outlet_Size,Outlet_Location_Type,Outlet_Type,Item_Outlet_Sales
0,FDA15,9.30,Low Fat,0.016047,Dairy,249.8092,OUT049,1999,Medium,Tier 1,Supermarket Type1,3735.1380
1,DRC01,5.92,Regular,0.019278,Soft Drinks,48.2692,OUT018,2009,Medium,Tier 3,Supermarket Type2,443.4228
2,FDN15,17.50,Low Fat,0.016760,Meat,141.6180,OUT049,1999,Medium,Tier 1,Supermarket Type1,2097.2700
3,FDX07,19.20,Regular,0.000000,Fruits and Vegetables,182.0950,OUT010,1998,NaN,Tier 3,Grocery Store,732.3800
4,NCD19,8.93,Low Fat,0.000000,Household,53.8614,OUT013,1987,High,Tier 3,Supermarket Type1,994.7052


----
# A simple cleaning process for data by removing duplicates and investigating any inconsistencies.

In [ ]:
# Check the dataframe information.
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8523 entries, 0 to 8522
Data columns (total 12 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Item_Identifier            8523 non-null   object 
 1   Item_Weight                7060 non-null   float64
 2   Item_Fat_Content           8523 non-null   object 
 3   Item_Visibility            8523 non-null   float64
 4   Item_Type                  8523 non-null   object 
 5   Item_MRP                   8523 non-null   float64
 6   Outlet_Identifier          8523 non-null   object 
 7   Outlet_Establishment_Year  8523 non-null   int64  
 8   Outlet_Size                6113 non-null   object 
 9   Outlet_Location_Type       8523 non-null   object 
 10  Outlet_Type                8523 non-null   object 
 11  Item_Outlet_Sales          8523 non-null   float64
dtypes: float64(4), int64(1), object(7)
memory usage: 799.2+ KB


In [ ]:
# Check for duplicates
df.duplicated().sum()

np.int64(0)

In [ ]:
# Check for null values
df.isna().sum()

,0
Item_Identifier,0
Item_Weight,1463
Item_Fat_Content,0
Item_Visibility,0
Item_Type,0
Item_MRP,0
Outlet_Identifier,0
Outlet_Establishment_Year,0
Outlet_Size,2410
Outlet_Location_Type,0


In [ ]:
# A simple modification to remove underscores and replace it with a space from all column names.
df = df.rename(columns = lambda x: x.replace('_', ' '))
df.head()

,Item Identifier,Item Weight,Item Fat Content,Item Visibility,Item Type,Item MRP,Outlet Identifier,Outlet Establishment Year,Outlet Size,Outlet Location Type,Outlet Type,Item Outlet Sales
0,FDA15,9.30,Low Fat,0.016047,Dairy,249.8092,OUT049,1999,Medium,Tier 1,Supermarket Type1,3735.1380
1,DRC01,5.92,Regular,0.019278,Soft Drinks,48.2692,OUT018,2009,Medium,Tier 3,Supermarket Type2,443.4228
2,FDN15,17.50,Low Fat,0.016760,Meat,141.6180,OUT049,1999,Medium,Tier 1,Supermarket Type1,2097.2700
3,FDX07,19.20,Regular,0.000000,Fruits and Vegetables,182.0950,OUT010,1998,NaN,Tier 3,Grocery Store,732.3800
4,NCD19,8.93,Low Fat,0.000000,Household,53.8614,OUT013,1987,High,Tier 3,Supermarket Type1,994.7052


In [ ]:
# this function checks each column in the dataframe
def check_column(df, column, cardinality_threshold=7, quasi_threshold=0.95):
    data_type = df[column].dtype
    null_values = df[column].isna().sum()
    null_pct = df[column].isna().mean() * 100

    if df[column].nunique() == 1:
        const_or_quasi = 'Constant'
    elif df[column].value_counts(normalize=True).iloc[0] >= quasi_threshold:
        const_or_quasi = 'Quasi-constant'
    else:
        const_or_quasi = 'Neither'

    if df[column].nunique() <= cardinality_threshold:
        cardinality = 'Low'
    else:
        cardinality = 'Medium or High'

    print(f'Column data type: {data_type}')
    print(10 * '--')
    print(f'Null values: {null_values} ({null_pct:.1f}%)')
    print(10 * '--')
    print(f'Constant or quasi-constant: {const_or_quasi}')
    print(10 * '--')
    print(f'Cardinality: {cardinality}')
    print(10 * '--')
    print('Value counts:')
    display(df[column].value_counts().sort_index())
    print(10 * '--')
    print('Descriptive statistics:')
    display(df[column].describe())


### check ``Item Identifier``
* The item identifier holds no meaningful value from a business or machine learning perspective

In [ ]:
# Item Identifier
check_column(df, 'Item Identifier')

Column data type: object
--------------------
Null values: 0 (0.0%)
--------------------
Constant or quasi-constant: Neither
--------------------
Cardinality: Medium or High
--------------------
Value counts:


,count
Item Identifier,
DRA12,6
DRA24,7
DRA59,8
DRB01,3
DRB13,5
...,...
NCZ30,7
NCZ41,5
NCZ42,5


--------------------
Descriptive statistics:


,Item Identifier
count,8523
unique,1559
top,FDW13
freq,10


### check ``Item Weight``
* Description: The weight of items ranges from 4.55 to 21.35.


In [ ]:
# Item Weight
check_column(df, 'Item Weight')

Column data type: float64
--------------------
Null values: 1463 (17.2%)
--------------------
Constant or quasi-constant: Neither
--------------------
Cardinality: Medium or High
--------------------
Value counts:


,count
Item Weight,
4.555,4
4.590,5
4.610,7
4.615,4
4.635,5
...,...
21.000,6
21.100,17
21.200,5


--------------------
Descriptive statistics:


,Item Weight
count,7060.000000
mean,12.857645
std,4.643456
min,4.555000
25%,8.773750
50%,12.600000
75%,16.850000
max,21.350000


### check ``Item Fat Content``
* Value Counts: Contains Low Fat, Regular, LF, low fat, and reg. This shows inconsistencies in categorization (e.g., 'Low Fat', 'LF', and 'low fat' likely refer to the same category, as do 'Regular' and 'reg').

In [ ]:
# Item Fat Content
check_column(df, 'Item Fat Content')

Column data type: object
--------------------
Null values: 0 (0.0%)
--------------------
Constant or quasi-constant: Neither
--------------------
Cardinality: Low
--------------------
Value counts:


,count
Item Fat Content,
LF,316
Low Fat,5089
Regular,2889
low fat,112
reg,117


--------------------
Descriptive statistics:


,Item Fat Content
count,8523
unique,5
top,Low Fat
freq,5089


In [ ]:
# Check for LF and low fat to Low Fat, reg to Regular
df['Item Fat Content'] = df['Item Fat Content'].replace({'LF': 'Low Fat', 'low fat': 'Low Fat', 'reg': 'Regular'})
df['Item Fat Content'].value_counts()

,count
Item Fat Content,
Low Fat,5517
Regular,3006


### check ``Item Visibility``
* Description: The visibility ranges from 0.0 to 0.328.


In [ ]:
# Item Visibility
check_column(df, 'Item Visibility')

Column data type: float64
--------------------
Null values: 0 (0.0%)
--------------------
Constant or quasi-constant: Neither
--------------------
Cardinality: Medium or High
--------------------
Value counts:


,count
Item Visibility,
0.000000,526
0.003575,1
0.003589,1
0.003598,1
0.003599,1
...,...
0.309390,1
0.311090,1
0.321115,1


--------------------
Descriptive statistics:


,Item Visibility
count,8523.000000
mean,0.066132
std,0.051598
min,0.000000
25%,0.026989
50%,0.053931
75%,0.094585
max,0.328391


### check ``Item Type``
* Top Value: 'Fruits and Vegetables' with 1232 occurrences.


In [ ]:
# Item Type
check_column(df, 'Item Type')

Column data type: object
--------------------
Null values: 0 (0.0%)
--------------------
Constant or quasi-constant: Neither
--------------------
Cardinality: Medium or High
--------------------
Value counts:


,count
Item Type,
Baking Goods,648
Breads,251
Breakfast,110
Canned,649
Dairy,682
Frozen Foods,856
Fruits and Vegetables,1232
Hard Drinks,214
Health and Hygiene,520


--------------------
Descriptive statistics:


,Item Type
count,8523
unique,16
top,Fruits and Vegetables
freq,1232


### check ``Item MRP``
* Description: Item Maximum Retail Price ranges from 31.29 to 266.88.


In [ ]:
# Item MRP
check_column(df, 'Item MRP')

Column data type: float64
--------------------
Null values: 0 (0.0%)
--------------------
Constant or quasi-constant: Neither
--------------------
Cardinality: Medium or High
--------------------
Value counts:


,count
Item MRP,
31.2900,1
31.4900,1
31.8900,1
31.9558,2
32.0558,1
...,...
266.1884,2
266.2884,1
266.5884,2


--------------------
Descriptive statistics:


,Item MRP
count,8523.000000
mean,140.992782
std,62.275067
min,31.290000
25%,93.826500
50%,143.012800
75%,185.643700
max,266.888400


### check ``Outlet Identifier``
* Top Value: 'OUT027' with 935 occurrences.


In [ ]:
# Outlet Identifier
check_column(df, 'Outlet Identifier')

Column data type: object
--------------------
Null values: 0 (0.0%)
--------------------
Constant or quasi-constant: Neither
--------------------
Cardinality: Medium or High
--------------------
Value counts:


,count
Outlet Identifier,
OUT010,555
OUT013,932
OUT017,926
OUT018,928
OUT019,528
OUT027,935
OUT035,930
OUT045,929
OUT046,930


--------------------
Descriptive statistics:


,Outlet Identifier
count,8523
unique,10
top,OUT027
freq,935


### check ``Outlet Establishment Year``
* Description: Outlets were established between 1985 and 2009.


In [ ]:
# Outlet Establishment Year
check_column(df, 'Outlet Establishment Year')

Column data type: int64
--------------------
Null values: 0 (0.0%)
--------------------
Constant or quasi-constant: Neither
--------------------
Cardinality: Medium or High
--------------------
Value counts:


,count
Outlet Establishment Year,
1985,1463
1987,932
1997,930
1998,555
1999,930
2002,929
2004,930
2007,926
2009,928


--------------------
Descriptive statistics:


,Outlet Establishment Year
count,8523.000000
mean,1997.831867
std,8.371760
min,1985.000000
25%,1987.000000
50%,1999.000000
75%,2004.000000
max,2009.000000


### check ``Outlet Size``
* Value Counts: Medium, Small, High.


In [ ]:
# Outlet Size
check_column(df, 'Outlet Size')

Column data type: object
--------------------
Null values: 2410 (28.3%)
--------------------
Constant or quasi-constant: Neither
--------------------
Cardinality: Low
--------------------
Value counts:


,count
Outlet Size,
High,932
Medium,2793
Small,2388


--------------------
Descriptive statistics:


,Outlet Size
count,6113
unique,3
top,Medium
freq,2793


### check ``Outlet Location Type``
* Value Counts: Tier 1, Tier 2, Tier 3.


In [ ]:
# Outlet Location Type
check_column(df, 'Outlet Location Type')

Column data type: object
--------------------
Null values: 0 (0.0%)
--------------------
Constant or quasi-constant: Neither
--------------------
Cardinality: Low
--------------------
Value counts:


,count
Outlet Location Type,
Tier 1,2388
Tier 2,2785
Tier 3,3350


--------------------
Descriptive statistics:


,Outlet Location Type
count,8523
unique,3
top,Tier 3
freq,3350


### check ``Outlet Type``
* Value Counts: Supermarket Type1, Grocery Store, Supermarket Type3, Supermarket Type2

In [ ]:
# Outlet Type
check_column(df, 'Outlet Type')

Column data type: object
--------------------
Null values: 0 (0.0%)
--------------------
Constant or quasi-constant: Neither
--------------------
Cardinality: Low
--------------------
Value counts:


,count
Outlet Type,
Grocery Store,1083
Supermarket Type1,5577
Supermarket Type2,928
Supermarket Type3,935


--------------------
Descriptive statistics:


,Outlet Type
count,8523
unique,4
top,Supermarket Type1
freq,5577


### check ``Item Outlet Sales``
* Description: Sales range from 33.29 to 13086.96. This is our target variable.


In [ ]:
# Item Outlet Sales
check_column(df, 'Item Outlet Sales')

Column data type: float64
--------------------
Null values: 0 (0.0%)
--------------------
Constant or quasi-constant: Neither
--------------------
Cardinality: Medium or High
--------------------
Value counts:


,count
Item Outlet Sales,
33.2900,2
33.9558,1
34.6216,1
35.2874,1
36.6190,2
...,...
10306.5840,1
10993.6896,1
11445.1020,1


--------------------
Descriptive statistics:


,Item Outlet Sales
count,8523.000000
mean,2181.288914
std,1706.499616
min,33.290000
25%,834.247400
50%,1794.331000
75%,3101.296400
max,13086.964800


# The preprocessing steps

In [ ]:
# the target column of our process is Item Outlet Sales
y = df['Item Outlet Sales']

X = df.drop(columns = ['Item Identifier', 'Item Outlet Sales'])

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state = 42)

In [ ]:
# X_train set
X_train.head()

,Item Weight,Item Fat Content,Item Visibility,Item Type,Item MRP,Outlet Identifier,Outlet Establishment Year,Outlet Size,Outlet Location Type,Outlet Type
4776,16.350,Low Fat,0.029565,Household,256.4646,OUT018,2009,Medium,Tier 3,Supermarket Type2
7510,15.250,Regular,0.000000,Snack Foods,179.7660,OUT018,2009,Medium,Tier 3,Supermarket Type2
5828,12.350,Regular,0.158716,Meat,157.2946,OUT049,1999,Medium,Tier 1,Supermarket Type1
5327,7.975,Low Fat,0.014628,Baking Goods,82.3250,OUT035,2004,Small,Tier 2,Supermarket Type1
4810,19.350,Low Fat,0.016645,Frozen Foods,120.9098,OUT045,2002,NaN,Tier 2,Supermarket Type1


# Numercial columns processing

In [ ]:
# numerical columns
num_cols = X_train.select_dtypes('number').columns
num_cols

Index(['Item Weight', 'Item Visibility', 'Item MRP',
       'Outlet Establishment Year'],
      dtype='object')

In [ ]:
# Check for mean and median and they are close to each other, so picking the mean is fine
X_train[num_cols].describe()

,Item Weight,Item Visibility,Item MRP,Outlet Establishment Year
count,5285.000000,6392.000000,6392.000000,6392.000000
mean,12.904458,0.066007,141.980400,1997.857165
std,4.637034,0.051131,62.629276,8.392300
min,4.555000,0.000000,31.290000,1985.000000
25%,8.895000,0.027027,94.146200,1987.000000
50%,12.650000,0.054152,144.110200,1999.000000
75%,17.000000,0.094618,186.900300,2004.000000
max,21.350000,0.328391,266.888400,2009.000000


In [ ]:
# num_imputer
num_imputer = SimpleImputer(strategy = 'mean')
num_imputer

SimpleImputer()

In [ ]:
# num_scaler
num_sclaer = StandardScaler()
num_sclaer

StandardScaler()

In [ ]:
# num_pipeline
num_pipe = make_pipeline(num_imputer, num_sclaer)
num_pipe

Pipeline(steps=[('simpleimputer', SimpleImputer()),
                ('standardscaler', StandardScaler())])

In [ ]:
# num_tuple
num_tuple = ('num_cols', num_pipe, num_cols)
num_tuple

('num_cols',
 Pipeline(steps=[('simpleimputer', SimpleImputer()),
                 ('standardscaler', StandardScaler())]),
 Index(['Item Weight', 'Item Visibility', 'Item MRP',
        'Outlet Establishment Year'],
       dtype='object'))

# Categorical (Nominal) columns preprocessing

In [ ]:
# ohe_cols
ohe_cols = X_train.select_dtypes('object').columns.drop(['Item Fat Content', 'Outlet Size', 'Outlet Location Type', 'Outlet Type'])
ohe_cols

Index(['Item Type', 'Outlet Identifier'], dtype='object')

In [ ]:
# one_imputer
one_imputer = SimpleImputer(strategy = 'most_frequent')
one_imputer

SimpleImputer(strategy='most_frequent')

In [ ]:
# ohe_encoder
ohe_encoder = OneHotEncoder(sparse_output = False, handle_unknown = 'ignore')
ohe_encoder

OneHotEncoder(handle_unknown='ignore', sparse_output=False)

In [ ]:
# ohe_pipe
ohe_pipe = make_pipeline(one_imputer, ohe_encoder)
ohe_pipe

Pipeline(steps=[('simpleimputer', SimpleImputer(strategy='most_frequent')),
                ('onehotencoder',
                 OneHotEncoder(handle_unknown='ignore', sparse_output=False))])

In [ ]:
# ohe_tuple
ohe_tuple = ('cat_cols', ohe_pipe, ohe_cols)
ohe_tuple

('cat_cols',
 Pipeline(steps=[('simpleimputer', SimpleImputer(strategy='most_frequent')),
                 ('onehotencoder',
                  OneHotEncoder(handle_unknown='ignore', sparse_output=False))]),
 Index(['Item Type', 'Outlet Identifier'], dtype='object'))

# Ordinal columns preprocessing

In [ ]:
# ord_columns
ord_cols = ['Item Fat Content', 'Outlet Size', 'Outlet Location Type', 'Outlet Type']
ord_cols

['Item Fat Content', 'Outlet Size', 'Outlet Location Type', 'Outlet Type']

In [ ]:
# ord_imputer
ord_imputer = SimpleImputer(strategy = 'most_frequent')
ord_imputer

SimpleImputer(strategy='most_frequent')

In [ ]:
# it has almost 30 percent null values, is it fine to use "most frequent"
round(df['Outlet Size'].isna().sum() / len(df['Outlet Size']), 2)

np.float64(0.28)

In [ ]:
# ord_encoding

categories = [
    ['Low Fat', 'Regular'],
    ['Small', 'Medium', 'High'],
    ['Tier 1', 'Tier 2', 'Tier 3'],
    ['Grocery Store', 'Supermarket Type1', 'Supermarket Type2', 'Supermarket Type3']]

ord_encdoer = OrdinalEncoder(categories = categories)
ord_encdoer

OrdinalEncoder(categories=[['Low Fat', 'Regular'], ['Small', 'Medium', 'High'],
                           ['Tier 1', 'Tier 2', 'Tier 3'],
                           ['Grocery Store', 'Supermarket Type1',
                            'Supermarket Type2', 'Supermarket Type3']])

In [ ]:
# ord_scaler
ord_sclaer = StandardScaler()
ord_sclaer

StandardScaler()

In [ ]:
# ord_pipe
ore_pipe = make_pipeline(ord_imputer, ord_encdoer, ord_sclaer)
ore_pipe

Pipeline(steps=[('simpleimputer', SimpleImputer(strategy='most_frequent')),
                ('ordinalencoder',
                 OrdinalEncoder(categories=[['Low Fat', 'Regular'],
                                            ['Small', 'Medium', 'High'],
                                            ['Tier 1', 'Tier 2', 'Tier 3'],
                                            ['Grocery Store',
                                             'Supermarket Type1',
                                             'Supermarket Type2',
                                             'Supermarket Type3']])),
                ('standardscaler', StandardScaler())])

In [ ]:
# ord_tuple
ord_tuple = ('ord_cols', ore_pipe, ord_cols)
ord_tuple

('ord_cols',
 Pipeline(steps=[('simpleimputer', SimpleImputer(strategy='most_frequent')),
                 ('ordinalencoder',
                  OrdinalEncoder(categories=[['Low Fat', 'Regular'],
                                             ['Small', 'Medium', 'High'],
                                             ['Tier 1', 'Tier 2', 'Tier 3'],
                                             ['Grocery Store',
                                              'Supermarket Type1',
                                              'Supermarket Type2',
                                              'Supermarket Type3']])),
                 ('standardscaler', StandardScaler())]),
 ['Item Fat Content', 'Outlet Size', 'Outlet Location Type', 'Outlet Type'])

In [ ]:
column_transformer = ColumnTransformer(transformers = [num_tuple, ohe_tuple, ord_tuple], verbose_feature_names_out = False)
column_transformer

ColumnTransformer(transformers=[('num_cols',
                                 Pipeline(steps=[('simpleimputer',
                                                  SimpleImputer()),
                                                 ('standardscaler',
                                                  StandardScaler())]),
                                 Index(['Item Weight', 'Item Visibility', 'Item MRP',
       'Outlet Establishment Year'],
      dtype='object')),
                                ('cat_cols',
                                 Pipeline(steps=[('simpleimputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('onehotencoder',
                                                  OneHotEncoder(handle_unk...
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('ordinalencoder',
                                                  OrdinalEncoder(categories=[['Low '
                                                                              'Fat',
                                                                              'Regular'],
                                                                             ['Small',
                                                                              'Medium',
                                                                              'High'],
                                                                             ['Tier '
                                                                              '1',
                                                                              'Tier '
                                                                              '2',
                                                                              'Tier '
                                                                              '3'],
                                                                             ['Grocery '
                                                                              'Store',
                                                                              'Supermarket '
                                                                              'Type1',
                                                                              'Supermarket '
                                                                              'Type2',
                                                                              'Supermarket '
                                                                              'Type3']])),
                                                 ('standardscaler',
                                                  StandardScaler())]),
                                 ['Item Fat Content', 'Outlet Size',
                                  'Outlet Location Type', 'Outlet Type'])],
                  verbose_feature_names_out=False)

In [ ]:
column_transformer.fit(X_train)

ColumnTransformer(transformers=[('num_cols',
                                 Pipeline(steps=[('simpleimputer',
                                                  SimpleImputer()),
                                                 ('standardscaler',
                                                  StandardScaler())]),
                                 Index(['Item Weight', 'Item Visibility', 'Item MRP',
       'Outlet Establishment Year'],
      dtype='object')),
                                ('cat_cols',
                                 Pipeline(steps=[('simpleimputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('onehotencoder',
                                                  OneHotEncoder(handle_unk...
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('ordinalencoder',
                                                  OrdinalEncoder(categories=[['Low '
                                                                              'Fat',
                                                                              'Regular'],
                                                                             ['Small',
                                                                              'Medium',
                                                                              'High'],
                                                                             ['Tier '
                                                                              '1',
                                                                              'Tier '
                                                                              '2',
                                                                              'Tier '
                                                                              '3'],
                                                                             ['Grocery '
                                                                              'Store',
                                                                              'Supermarket '
                                                                              'Type1',
                                                                              'Supermarket '
                                                                              'Type2',
                                                                              'Supermarket '
                                                                              'Type3']])),
                                                 ('standardscaler',
                                                  StandardScaler())]),
                                 ['Item Fat Content', 'Outlet Size',
                                  'Outlet Location Type', 'Outlet Type'])],
                  verbose_feature_names_out=False)

In [ ]:
X_train_tr = column_transformer.transform(X_train)
X_train_tr

,Item Weight,Item Visibility,Item MRP,Outlet Establishment Year,Item Type_Baking Goods,Item Type_Breads,Item Type_Breakfast,Item Type_Canned,Item Type_Dairy,Item Type_Frozen Foods,...,Outlet Identifier_OUT019,Outlet Identifier_OUT027,Outlet Identifier_OUT035,Outlet Identifier_OUT045,Outlet Identifier_OUT046,Outlet Identifier_OUT049,Item Fat Content,Outlet Size,Outlet Location Type,Outlet Type
4776,0.817249,-0.712775,1.828109,1.327849,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,-0.740321,0.287374,1.084948,0.983572
7510,0.556340,-1.291052,0.603369,1.327849,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.350766,0.287374,1.084948,0.983572
5828,-0.131512,1.813319,0.244541,0.136187,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,1.350766,0.287374,-1.384777,-0.263600
5327,-1.169219,-1.004931,-0.952591,0.732018,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,-0.740321,-1.384048,-0.149914,-0.263600
4810,1.528819,-0.965484,-0.336460,0.493686,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,1.0,0.0,0.0,-0.740321,0.287374,-0.149914,-0.263600
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5734,-0.832409,4.309657,-0.044657,0.017021,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.350766,0.287374,1.084948,-1.510771
5191,0.639356,1.008625,-1.058907,1.089517,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,-0.740321,0.287374,-0.149914,-0.263600
5390,1.113736,-0.920527,1.523027,0.493686,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,-0.740321,0.287374,-0.149914,-0.263600
860,1.766009,-0.227755,-0.383777,1.089517,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,-0.740321,0.287374,-0.149914,-0.263600


In [ ]:
X_test_tr = column_transformer.transform(X_test)
X_test_tr

,Item Weight,Item Visibility,Item MRP,Outlet Establishment Year,Item Type_Baking Goods,Item Type_Breads,Item Type_Breakfast,Item Type_Canned,Item Type_Dairy,Item Type_Frozen Foods,...,Outlet Identifier_OUT019,Outlet Identifier_OUT027,Outlet Identifier_OUT035,Outlet Identifier_OUT045,Outlet Identifier_OUT046,Outlet Identifier_OUT049,Item Fat Content,Outlet Size,Outlet Location Type,Outlet Type
7503,3.310089e-01,-0.776646,-0.998816,-1.293807,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,-0.740321,1.958796,1.084948,-0.263600
2957,-1.179892e+00,0.100317,-1.585194,-0.102145,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,-0.740321,-1.384048,-1.384777,-0.263600
7031,3.784469e-01,-0.482994,-1.595784,0.136187,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,1.350766,0.287374,-1.384777,-0.263600
1084,4.213344e-16,-0.415440,0.506592,-1.532139,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,1.350766,0.287374,1.084948,2.230744
856,-6.426567e-01,-1.047426,0.886725,0.732018,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,1.350766,-1.384048,-0.149914,-0.263600
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4753,1.113736e+00,-1.134688,0.473646,-1.293807,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,-0.740321,1.958796,1.084948,-0.263600
4836,-6.426567e-01,-1.291052,0.018124,1.089517,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,-0.740321,0.287374,-0.149914,-0.263600
8064,-1.139570e+00,1.218324,1.093980,0.493686,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,-0.740321,0.287374,-0.149914,-0.263600
4418,-1.497727e+00,-0.778096,-0.366800,0.136187,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,-0.740321,0.287374,-1.384777,-0.263600


In [ ]:
# evaluation functions

def regression_metrics(y_true, y_pred, label='', verbose=True, output_dict=False):
    # Get metrics
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = root_mean_squared_error(y_true, y_pred)
    r_squared = r2_score(y_true, y_pred)

    if verbose == True:
        # Print Result with Label and Header
        header = "-"*60
        print(header, f"Regression Metrics: {label}", header, sep='\n')
        print(f"- MAE = {mae:,.3f}")
        print(f"- MSE = {mse:,.3f}")
        print(f"- RMSE = {rmse:,.3f}")
        print(f"- R^2 = {r_squared:,.3f}")

    if output_dict == True:
        metrics = {'Label': label, 'MAE': mae,
                   'MSE': mse, 'RMSE': rmse, 'R^2': r_squared}
        return metrics


######################################################################################

def evaluate_regression(reg, X_train, y_train, X_test, y_test, verbose=True,
                        output_frame=False):
    # Get predictions for training data
    y_train_pred = reg.predict(X_train)

    # Call the helper function to obtain regression metrics for training data
    results_train = regression_metrics(y_train, y_train_pred, verbose=verbose,
                                       output_dict=output_frame,
                                       label='Training Data')
    print()

    # Get predictions for test data
    y_test_pred = reg.predict(X_test)

    # Call the helper function to obtain regression metrics for test data
    results_test = regression_metrics(y_test, y_test_pred, verbose=verbose,
                                      output_dict=output_frame,
                                      label='Test Data')

    # Store results in a dataframe if output_frame is True
    if output_frame:
        results_df = pd.DataFrame([results_train, results_test])
        # Set the label as the index
        results_df = results_df.set_index('Label')
        # Set index.name to none to get a cleaner looking result
        results_df.index.name = None
        # Return the dataframe
        return results_df.round(3)



### Project 1 - Part 6 (Core)

In [ ]:
# build a linear regression model

In [ ]:
# lin_reg = LinearRegression()
# lin_reg

In [ ]:
# fit the model
# lin_reg.fit(X_train_tr, y_train)

In [ ]:
# evaluate_function = evaluate_regression(lin_reg, X_train_tr, y_train, X_test_tr, y_test)
# evaluate_function

### My interpretation of the LinearRegression model is:
> The training R² (0.562) and test R² (0.567) are nearly identical, indicating that the model is **niether** overfit nor underfit and generalizes consistently across both sets. However, with only ~56% of the variance in sales prices explained, the model is considered a weak fit for this task. This suggests that Linear Regression is too simple to capture the underlying complexity of housing price patterns, pointing to high bias rather than a variance problem. A more powerful model or additional feature engineering would likely yield significantly better results.




In [ ]:
# build a random forest model
random_forest = RandomForestRegressor()
random_forest

RandomForestRegressor()

In [ ]:
random_forest.fit(X_train_tr, y_train)

RandomForestRegressor()

In [ ]:
evaluate_function = evaluate_regression(random_forest, X_train_tr, y_train, X_test_tr, y_test)
evaluate_function

------------------------------------------------------------
Regression Metrics: Training Data
------------------------------------------------------------
- MAE = 297.616
- MSE = 184,304.463
- RMSE = 429.307
- R^2 = 0.938

------------------------------------------------------------
Regression Metrics: Test Data
------------------------------------------------------------
- MAE = 776.827
- MSE = 1,236,730.292
- RMSE = 1,112.084
- R^2 = 0.552


### My interpretation of the RandomForest model is:

> The Random Forest model achieves a training R² of 0.938, which appears impressive at first glance. However, the test R² drops drastically to 0.556 — nearly identical to the Linear Regression result — **which is a textbook sign of overfitting**.

> The model has essentially memorized the training data rather than learning generalizable patterns, and this is reflected in the error metrics as well, where the test RMSE (1,106) is more than double the training RMSE (429). To address this, hyperparameter tuning — such as limiting max_depth, increasing min_samples_split, or reducing n_estimators — would be the recommended next step.

---

### Which model has the best test score?
>  the difference is so marginal (0.567 vs 0.556) that it's practically negligible — both models perform the same on test data. Declaring Linear Regression a "winner" by that tiny gap would be misleading.

### Hyperparameter Tuning with GridSearchCV

In [ ]:
from sklearn.model_selection import GridSearchCV

# Define the parameter grid for GridSearchCV
param_grid = {
    'n_estimators': [100, 200, 300],  # Number of trees in the forest
    'max_depth': [None, 10, 20]       # Maximum depth of the tree
}

# Initialize a RandomForestRegressor instance
rf_model_grid = RandomForestRegressor(random_state=42)

# Initialize GridSearchCV
grid_search = GridSearchCV(
    estimator=rf_model_grid,
    param_grid=param_grid,
    cv=5, # 5-fold cross-validation
    scoring='neg_mean_squared_error', # Optimize for lower MSE
    n_jobs=-1, # Use all available cores
    verbose=1
)

# Fit GridSearchCV to the training data
grid_search.fit(X_train_tr, y_train)

# Get the best parameters and best score
best_params = grid_search.best_params_
best_score = -grid_search.best_score_ # Convert back to positive MSE

print(f"Best parameters found: {best_params}")
print(f"Best MSE on training data (cross-validated): {best_score:,.3f}")

Fitting 5 folds for each of 9 candidates, totalling 45 fits
Best parameters found: {'max_depth': 10, 'n_estimators': 200}
Best MSE on training data (cross-validated): 1,218,343.129


### Fit and Evaluate the Tuned Random Forest Model

In [ ]:
# Initialize the final model with the best parameters
final_rf_model = RandomForestRegressor(**best_params, random_state=42)

# Fit the final model on the entire training set
final_rf_model.fit(X_train_tr, y_train)

# Evaluate the final model
print("\nEvaluating Tuned Random Forest Model:")
evaluate_regression(final_rf_model, X_train_tr, y_train, X_test_tr, y_test)


Evaluating Tuned Random Forest Model:
------------------------------------------------------------
Regression Metrics: Training Data
------------------------------------------------------------
- MAE = 642.699
- MSE = 823,249.372
- RMSE = 907.331
- R^2 = 0.722

------------------------------------------------------------
Regression Metrics: Test Data
------------------------------------------------------------
- MAE = 739.600
- MSE = 1,132,330.463
- RMSE = 1,064.110
- R^2 = 0.590


### The question after using GridSearchCV, did the performance increase?

> Yes, but modestly. The tuned model reduced overfitting significantly — the training R² dropped from 0.938 to 0.722, which is actually a good sign — and the test R² improved slightly from 0.552 to 0.590. The model now generalizes better, but the overall test performance is still not a major leap forward.



---



### Which model would you implement?
> Recommended Model: Tuned Random Forest
Among all tested models, the Tuned Random Forest is the recommended model, as it achieves the best test R² of 0.590 — outperforming both Linear Regression (0.567) and the untuned Random Forest (0.552) on unseen data, while also showing significantly reduced overfitting compared to the untuned version.

### R-squared explained for non-technincal stakeholders:

> R² Interpretation (for non-technical stakeholders):
Our model can correctly explain about 59% of the variation in sales prices. In other words, if a house's price changes, our model can account for roughly 59 out of 100 of those changes based on the features provided.


### Another metrics explained for non-technincal stakeholders:
> Selected Metric: MAE (Mean Absolute Error)
We selected MAE as our supporting metric because it is the most intuitive for stakeholders — it tells us that, on average, our model's price predictions are off by approximately $739. Unlike MSE or RMSE, MAE is not heavily influenced by outliers and is expressed in the same unit as the target variable (price), making it easy to communicate.
Overfitting/Underfitting Assessment:


### To what extent is this model overfit/underfit?
> The tuned model has a training R² of 0.722 and a test R² of 0.590. While a gap still exists, it is considerably smaller than the untuned Random Forest (0.938 vs 0.552), indicating that hyperparameter tuning successfully reduced overfitting. The model is moderately overfit, but it represents the best balance between training performance and generalization achieved across all tested models.